# MWAA Observability with Amazon OpenSearch Service and MCP

This notebook demonstrates an end-to-end observability solution for ETL workloads orchestrated by Amazon MWAA. You will:

1. **Trigger the ETL DAG** to generate logs across EC2, MWAA, and Glue.
2. **Connect to OpenSearch** and configure IAM role mappings.
3. **Stream CloudWatch logs into OpenSearch** via subscription filters and a Lambda function.
4. **Register a Claude LLM connector** in OpenSearch for intelligent log analysis.
5. **Query logs using an AI agent** powered by the OpenSearch MCP server and Strands Agents.

---
## 1. Prerequisites

Install required libraries and import modules.

In [1]:
!pip install opensearch-py requests-aws4auth -q
print("Installs completed.")

Installs completed.


In [29]:
import boto3
import json
import time
import io
import zipfile
import base64
import hashlib


import requests as http_requests
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth

# Tracking variables for idempotent connector/model creation
llm_connector_id = ""
llm_model_id = ""

print("Imports completed.")

Imports completed.


### 1.1 Load CloudFormation Stack Outputs

Retrieve resource names and ARNs provisioned by the workshop CloudFormation stacks.

In [3]:
session = boto3.Session()
region = session.region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
cfn = boto3.client("cloudformation")


def get_cfn_outputs(stack_name: str) -> dict:
    """Return a dict of OutputKey -> OutputValue for a CloudFormation stack."""
    stacks = cfn.describe_stacks(StackName=stack_name)["Stacks"]
    return {o["OutputKey"]: o["OutputValue"] for o in stacks[0]["Outputs"]}


# Main OpenSearch stack
outputs = get_cfn_outputs("opensearch-cfn")
aos_host = outputs["OpenSearchDomainEndpoint"]
bedrock_inf_iam_role = outputs["BedrockBatchInferenceRole"]
bedrock_inf_iam_role_arn = outputs["BedrockBatchInferenceRoleArn"]
notebook_iam_role_arn = outputs["NotebookRoleArn"]

# AgentCore MCP server stack
outputs_mcp = get_cfn_outputs("agentcore-mcp-server")
agentcore_iam_role_arn = outputs_mcp["AgentExecutionRoleArn"]
agentcore_token_endpoint = outputs_mcp["TokenEndpoint"]
agentcore_mcp_endpoint = outputs_mcp["MCPServerEndpoint"]
agentcore_cognito_secret = outputs_mcp["CognitoSecret"]

# ETL stack
etl_outputs = get_cfn_outputs("etl")

print(f"Region: {region}")
print(f"OpenSearch endpoint: {aos_host}")
print(f"MWAA environment: {etl_outputs.get('MWAAEnvironmentName', 'N/A')}")

Region: us-west-2
OpenSearch endpoint: search-opensearchservi-c15b5upchbkh-hz6kxevkpkeyculkygn6m54ccu.us-west-2.es.amazonaws.com
MWAA environment: etl-mwaa


---
## 2. Connect to OpenSearch

### 2.1 Authenticate with internal credentials

Retrieve the OpenSearch admin credentials from Secrets Manager and establish an authenticated connection.

In [4]:
secrets_client = boto3.client("secretsmanager")
aos_credentials = json.loads(
    secrets_client.get_secret_value(SecretId=outputs["OpenSearchSecret"])["SecretString"]
)

aos_client = OpenSearch(
    hosts=[f"https://{aos_host}"],
    http_auth=(aos_credentials["username"], aos_credentials["password"]),
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)

print(f"Connected to OpenSearch: {aos_host}")

Connected to OpenSearch: search-opensearchservi-c15b5upchbkh-hz6kxevkpkeyculkygn6m54ccu.us-west-2.es.amazonaws.com


### 2.2 Map IAM roles to OpenSearch backend roles

Grant `all_access` to the notebook role and the AgentCore execution role so both can read/write indices.

In [5]:
role_mapping = {
    "backend_roles": [notebook_iam_role_arn, agentcore_iam_role_arn],
    "users": [aos_credentials["username"]],
}

response = aos_client.security.create_role_mapping(role="all_access", body=role_mapping)
print(f"Role mapping updated: {response}")

Role mapping updated: {'status': 'OK', 'message': "'all_access' updated."}


### 2.3 Switch to IAM-based authentication

For the remainder of this notebook we authenticate using the notebook's IAM role via SigV4. If you encounter expired-token errors later, re-run this cell to refresh credentials.

In [6]:
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    "es",
    session_token=credentials.token,
)

aos_client = OpenSearch(
    hosts=[f"https://{aos_host}"],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=60,
)

print(f"Authenticated via IAM: {aos_client}")

Authenticated via IAM: <OpenSearch([{'host': 'search-opensearchservi-c15b5upchbkh-hz6kxevkpkeyculkygn6m54ccu.us-west-2.es.amazonaws.com', 'port': 443, 'use_ssl': True}])>


In [7]:
# Persist connection variables for use in subsequent cells and notebooks
OPENSEARCH_URL = f"https://{aos_host}"
AWS_ACCESS_KEY_ID = credentials.access_key
AWS_SECRET_ACCESS_KEY = credentials.secret_key
AWS_SESSION_TOKEN = credentials.token
AWS_REGION = region
MCP_ENDPOINT = agentcore_mcp_endpoint
MCP_TOKEN_ENDPOINT = agentcore_token_endpoint
COGNITO_SECRET = agentcore_cognito_secret

%store OPENSEARCH_URL AWS_ACCESS_KEY_ID AWS_SECRET_ACCESS_KEY AWS_SESSION_TOKEN AWS_REGION MCP_ENDPOINT MCP_TOKEN_ENDPOINT COGNITO_SECRET

Stored 'OPENSEARCH_URL' (str)
Stored 'AWS_ACCESS_KEY_ID' (str)
Stored 'AWS_SECRET_ACCESS_KEY' (str)
Stored 'AWS_SESSION_TOKEN' (str)
Stored 'AWS_REGION' (str)
Stored 'MCP_ENDPOINT' (str)
Stored 'MCP_TOKEN_ENDPOINT' (str)
Stored 'COGNITO_SECRET' (str)


---
## 3. Stream CloudWatch Logs into OpenSearch

This section creates a Lambda function and CloudWatch subscription filters that stream logs from all ETL-related log groups (EC2, MWAA, Glue, OpenSearch) into OpenSearch indices prefixed with `cwl-`.

In [10]:
logs_client = boto3.client("logs")
iam_client = boto3.client("iam")
lambda_client = boto3.client("lambda")

opensearch_endpoint = f"https://{aos_host}"

# Log group prefixes covering EC2, MWAA, Glue, and OpenSearch
LOG_GROUP_PREFIXES = [
    "/observability-blog/ec2-etl",
    "/airflow/",
    "airflow-",
    "/aws-glue/jobs/",
    "/aws/mwaa-serverless/",
    "/aws/opensearch/",
]

# Discover matching log groups
target_log_groups = []
for prefix in LOG_GROUP_PREFIXES:
    paginator = logs_client.get_paginator("describe_log_groups")
    for page in paginator.paginate(logGroupNamePrefix=prefix):
        target_log_groups.extend(lg["logGroupName"] for lg in page["logGroups"])

print(f"Discovered {len(target_log_groups)} log groups:")
for lg in target_log_groups:
    print(f"  - {lg}")

Discovered 12 log groups:
  - /observability-blog/ec2-etl
  - /airflow/etl-mwaa/Task
  - airflow-etl-mwaa-DAGProcessing
  - airflow-etl-mwaa-Scheduler
  - airflow-etl-mwaa-Task
  - airflow-etl-mwaa-WebServer
  - airflow-etl-mwaa-Worker
  - /aws-glue/jobs/ObservabilityBlogAggregation
  - /aws-glue/jobs/ObservabilityBlogETL
  - /aws-glue/jobs/error
  - /aws-glue/jobs/logs-v2
  - /aws/mwaa-serverless/etl-serverless-agg


In [11]:
# --- IAM Role for the Lambda function ---
LAMBDA_ROLE_NAME = "CWLtoOpenSearchLambdaRole"
LAMBDA_FUNCTION_NAME = "CWLtoOpenSearch"

lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

lambda_permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["es:ESHttpPost", "es:ESHttpPut"],
            "Resource": f"arn:aws:es:{region}:{account_id}:domain/{outputs['OpenSearchDomainName']}/*",
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": f"arn:aws:logs:{region}:{account_id}:*",
        },
    ],
}

# Create or reuse the Lambda execution role
try:
    role_response = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)
    lambda_role_arn = role_response["Role"]["Arn"]
    print(f"IAM role already exists: {LAMBDA_ROLE_NAME}")
except iam_client.exceptions.NoSuchEntityException:
    role_response = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(lambda_trust_policy),
        Description="Lambda role for streaming CloudWatch Logs to OpenSearch",
    )
    lambda_role_arn = role_response["Role"]["Arn"]
    print(f"Created IAM role: {LAMBDA_ROLE_NAME}")

iam_client.put_role_policy(
    RoleName=LAMBDA_ROLE_NAME,
    PolicyName="CWLtoOpenSearchLambdaPolicy",
    PolicyDocument=json.dumps(lambda_permissions_policy),
)
print(f"Lambda Role ARN: {lambda_role_arn}")
print("Waiting 10s for IAM propagation...")
time.sleep(10)

IAM role already exists: CWLtoOpenSearchLambdaRole
Lambda Role ARN: arn:aws:iam::344824510306:role/CWLtoOpenSearchLambdaRole
Waiting 10s for IAM propagation...


In [12]:
# --- Lambda function code ---
LAMBDA_CODE = '''
import base64
import gzip
import json
import os
import datetime
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

OPENSEARCH_ENDPOINT = os.environ["OPENSEARCH_ENDPOINT"]
REGION = os.environ["AWS_REGION"]


def get_credentials():
    return boto3.Session().get_credentials().get_frozen_credentials()


def sign_request(method, url, body, region, service="es"):
    creds = get_credentials()
    request = AWSRequest(method=method, url=url, data=body, headers={"Content-Type": "application/json"})
    SigV4Auth(creds, service, region).add_auth(request)
    return dict(request.headers)


def handler(event, context):
    compressed = base64.b64decode(event["awslogs"]["data"])
    payload = json.loads(gzip.decompress(compressed))

    if payload.get("messageType") == "CONTROL_MESSAGE":
        return {"statusCode": 200, "body": "Control message ignored"}

    log_group = payload.get("logGroup", "unknown")
    log_stream = payload.get("logStream", "unknown")
    log_events = payload.get("logEvents", [])

    if not log_events:
        return {"statusCode": 200, "body": "No log events"}

    today = datetime.datetime.utcnow().strftime("%Y.%m.%d")
    index_name = f"cwl-{today}"

    bulk_body = ""
    for evt in log_events:
        doc = {
            "@timestamp": datetime.datetime.utcfromtimestamp(evt["timestamp"] / 1000).isoformat() + "Z",
            "message": evt.get("message", ""),
            "log_group": log_group,
            "log_stream": log_stream,
            "id": evt.get("id", ""),
        }
        try:
            parsed = json.loads(evt["message"])
            if isinstance(parsed, dict):
                doc.update(parsed)
        except (json.JSONDecodeError, TypeError):
            pass
        bulk_body += json.dumps({"index": {"_index": index_name}}) + "\\n"
        bulk_body += json.dumps(doc) + "\\n"

    url = f"{OPENSEARCH_ENDPOINT}/_bulk"
    headers = sign_request("POST", url, bulk_body, REGION)
    req = Request(url, data=bulk_body.encode("utf-8"), headers=headers, method="POST")
    try:
        with urlopen(req) as resp:
            response_body = json.loads(resp.read())
            if response_body.get("errors"):
                print(f"Bulk indexing errors: {json.dumps(response_body)}")
            return {"statusCode": 200, "body": f"Indexed {len(log_events)} events to {index_name}"}
    except HTTPError as e:
        print(f"OpenSearch bulk request failed: {e.code} {e.read().decode()}")
        raise
'''

# Package as zip
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", LAMBDA_CODE)
zip_bytes = zip_buffer.getvalue()

# Create or update Lambda function
try:
    lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
    lambda_client.update_function_code(FunctionName=LAMBDA_FUNCTION_NAME, ZipFile=zip_bytes)
    time.sleep(5)
    lambda_client.update_function_configuration(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Environment={"Variables": {"OPENSEARCH_ENDPOINT": opensearch_endpoint}},
    )
    print(f"Updated Lambda function: {LAMBDA_FUNCTION_NAME}")
except lambda_client.exceptions.ResourceNotFoundException:
    lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime="python3.12",
        Role=lambda_role_arn,
        Handler="index.handler",
        Code={"ZipFile": zip_bytes},
        Timeout=60,
        MemorySize=256,
        Environment={"Variables": {"OPENSEARCH_ENDPOINT": opensearch_endpoint}},
    )
    print(f"Created Lambda function: {LAMBDA_FUNCTION_NAME}")

lambda_arn = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)["Configuration"]["FunctionArn"]
print(f"Lambda ARN: {lambda_arn}")

Created Lambda function: CWLtoOpenSearch
Lambda ARN: arn:aws:lambda:us-west-2:344824510306:function:CWLtoOpenSearch


In [30]:
# Grant CloudWatch Logs permission to invoke the Lambda
for log_group in target_log_groups:
    statement_id = statement_id = f"CWL{hashlib.md5(log_group.encode()).hexdigest()[:12]}"

    try:
        lambda_client.add_permission(
            FunctionName=LAMBDA_FUNCTION_NAME,
            StatementId=statement_id,
            Action="lambda:InvokeFunction",
            Principal="logs.amazonaws.com",
            SourceArn=f"arn:aws:logs:{region}:{account_id}:log-group:{log_group}:*",
            SourceAccount=account_id,
        )
    except lambda_client.exceptions.ResourceConflictException:
        pass  # Permission already exists

# Create subscription filters.
# The Lambda resource policy added above is eventually consistent: CloudWatch Logs
# runs a synchronous "can I invoke this function?" check when the filter is created,
# and that check can fail with InvalidParameterException ("Could not execute the
# lambda function") until the policy has propagated. Retry each group with backoff
# instead of relying on a fixed sleep, so a slow propagation doesn't cause a failure.
FILTER_NAME = "OpenSearchObservabilityFilter"
MAX_ATTEMPTS = 6
success_count = 0
for log_group in target_log_groups:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            logs_client.put_subscription_filter(
                logGroupName=log_group,
                filterName=FILTER_NAME,
                filterPattern="",
                destinationArn=lambda_arn,
            )
            print(f"  \u2705 {log_group}")
            success_count += 1
            break
        except logs_client.exceptions.InvalidParameterException as e:
            # Transient: Lambda invoke permission not yet enforceable. Back off and retry.
            if attempt == MAX_ATTEMPTS:
                print(f"  \u274c {log_group}: {e} (gave up after {MAX_ATTEMPTS} attempts)")
            else:
                time.sleep(2 ** attempt)  # 2, 4, 8, 16, 32 seconds
        except Exception as e:
            print(f"  \u274c {log_group}: {e}")
            break

print(f"\nCreated {success_count}/{len(target_log_groups)} subscription filters.")
print(f"Index pattern: cwl-YYYY.MM.DD")

  ✅ /observability-blog/ec2-etl
  ✅ /airflow/etl-mwaa/Task
  ✅ airflow-etl-mwaa-DAGProcessing
  ✅ airflow-etl-mwaa-Scheduler
  ✅ airflow-etl-mwaa-Task
  ✅ airflow-etl-mwaa-WebServer
  ✅ airflow-etl-mwaa-Worker
  ✅ /aws-glue/jobs/ObservabilityBlogAggregation
  ✅ /aws-glue/jobs/ObservabilityBlogETL
  ✅ /aws-glue/jobs/error
  ✅ /aws-glue/jobs/logs-v2
  ✅ /aws/mwaa-serverless/etl-serverless-agg

Created 12/12 subscription filters.
Index pattern: cwl-YYYY.MM.DD


In [14]:
# Map Lambda execution role to OpenSearch all_access
# The Security API requires the internal admin user, not IAM auth.
aos_admin_client = OpenSearch(
    hosts=[f"https://{aos_host}"],
    http_auth=(aos_credentials["username"], aos_credentials["password"]),
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
)

try:
    current_mapping = aos_admin_client.security.get_role_mapping(role="all_access")
    existing_backend_roles = current_mapping.get("all_access", {}).get("backend_roles", [])
    existing_users = current_mapping.get("all_access", {}).get("users", [])
except Exception:
    existing_backend_roles = []
    existing_users = []
 
actual_lambda_role_arn = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)["Configuration"]["Role"]

if lambda_role_arn not in existing_backend_roles:
    existing_backend_roles.append(lambda_role_arn)

response = aos_admin_client.security.create_role_mapping(
    role="all_access",
    body={"backend_roles": existing_backend_roles, "users": existing_users},
)
print(f"Lambda role added to all_access: {response}")

Lambda role added to all_access: {'status': 'OK', 'message': "'all_access' updated."}


---
## 4. Trigger the ETL DAG

Run this cell to trigger the ETL DAG and generate logs. Set `USE_SERVERLESS` to choose the runtime:

| Mode | DAG | Description |
|------|-----|-------------|
| `False` (default) | `observability_etl_dag` | Provisioned MWAA — runs Glue + EC2 in parallel |
| `True` | `observability_blog_aggregation` | MWAA Serverless — runs Glue aggregation job |

In [15]:
USE_SERVERLESS = False

if USE_SERVERLESS:
    print("\U0001f680 Triggering DAG via MWAA Serverless...")
    mwaa_serverless_client = boto3.client("mwaa-serverless")
    workflow_arn = etl_outputs["MWAAServerlessWorkflowArn"]

    response = mwaa_serverless_client.start_workflow_execution(WorkflowArn=workflow_arn)
    execution_id = response.get("ExecutionId", "N/A")

    print(f"  Workflow ARN: {workflow_arn}")
    print(f"  DAG: observability_blog_aggregation")
    print(f"  Execution ID: {execution_id}")
    print("  Status: Triggered \u2705")

else:
    print("\U0001f680 Triggering DAG via Provisioned MWAA...")
    mwaa_client = boto3.client("mwaa")
    mwaa_env_name = etl_outputs["MWAAEnvironmentName"]
    dag_id = "observability_etl_dag"

    # Get CLI token
    cli_token_response = mwaa_client.create_cli_token(Name=mwaa_env_name)
    cli_token = cli_token_response["CliToken"]
    web_server_hostname = cli_token_response["WebServerHostname"]

    url = f"https://{web_server_hostname}/aws_mwaa/cli"
    headers = {
        "Authorization": f"Bearer {cli_token}",
        "Content-Type": "text/plain",
    }

    # Unpause then trigger
    http_requests.post(url, headers=headers, data=f"dags unpause {dag_id}")
    trigger_response = http_requests.post(url, headers=headers, data=f"dags trigger {dag_id}")

    if trigger_response.status_code == 200:
        output = base64.b64decode(trigger_response.json().get("stdout", "")).decode()
        stderr = base64.b64decode(trigger_response.json().get("stderr", "")).decode()
        print(f"  MWAA Environment: {mwaa_env_name}")
        print(f"  DAG: {dag_id}")
        print(f"  Response: {output}")
        if stderr:
            print(f"  Warnings: {stderr}")
        print("  Status: Triggered \u2705")
    else:
        print(f"  \u274c Failed to trigger DAG. HTTP {trigger_response.status_code}")
        print(f"  Response: {trigger_response.text}")

🚀 Triggering DAG via Provisioned MWAA...
  MWAA Environment: etl-mwaa
  DAG: observability_etl_dag
  Response: [2026-06-22T18:15:44.559+0000] {__init__.py:43} INFO - Loaded API auth backend: airflow.api.auth.backend.session
     |                     |                     |                     |                     |          |                  | last_scheduling_dec |                      |          |            |       
conf | dag_id              | dag_run_id          | data_interval_start | data_interval_end   | end_date | external_trigger | ision               | logical_date         | run_type | start_date | state 
=====+=====================+=====================+=====================+=====================+==========+==================+=====================+======================+==========+============+=======
{}   | observability_etl_d | manual__2026-06-22T | 2026-06-22          | 2026-06-22          | None     | True             | None                | 2026-06-22           | man

### 4.1 Verify DAG has stopped running
This cell polls the DAG to see if it has completed succesfully, failed, or is still running.  You should get the response the DAG failed.

In [16]:
import time

while True:
    cli_token_response = mwaa_client.create_cli_token(Name=mwaa_env_name)
    url = f"https://{cli_token_response['WebServerHostname']}/aws_mwaa/cli"
    headers = {
        "Authorization": f"Bearer {cli_token_response['CliToken']}",
        "Content-Type": "text/plain",
    }

    response = http_requests.post(url, headers=headers, data=f"dags list-runs -d {dag_id}")
    stdout = base64.b64decode(response.json().get("stdout", "")).decode()

    lines = [l for l in stdout.strip().split("\n") if dag_id in l]
    if lines:
        state = [c.strip() for c in lines[0].split("|")][2]
        if state == "success":
            print("✅ DAG succeeded.")
            break
        elif state == "failed":
            print("❌ DAG failed.")
            break
        else:
            print(f"⏳ DAG is {state}...")
    else:
        print("⏳ No runs found yet...")

    time.sleep(30)


⏳ DAG is running...
⏳ DAG is running...
❌ DAG failed.


### 4.2 Verify log ingestion

After waiting for logs to flow, confirm that documents have been indexed. Update the date in the index name to today's date if needed.

In [17]:
from datetime import datetime, timezone

today = datetime.now(timezone.utc).strftime("%Y.%m.%d")
index_name = f"cwl-{today}"

try:
    res = aos_client.search(index=index_name, body={"query": {"match_all": {}}})
    print(f"Index: {index_name} — Documents found: {res['hits']['total']['value']}")
except Exception as e:
    print(f"Index {index_name} not found yet. Logs may still be streaming. Error: {e}")

Index: cwl-2026.06.22 — Documents found: 255


---
## 5. Register Claude LLM Connector in OpenSearch

Create an OpenSearch ML connector to Amazon Bedrock Claude for intelligent log analysis.

In [18]:
CLAUDE_MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

if not llm_connector_id:
    connector_payload = {
        "name": "Amazon Bedrock Connector: Claude",
        "description": "Connector to Amazon Bedrock Claude for log analysis",
        "version": 1,
        "protocol": "aws_sigv4",
        "credential": {
            "roleArn": f"arn:aws:iam::{account_id}:role/{bedrock_inf_iam_role}"
        },
        "parameters": {
            "region": region,
            "service_name": "bedrock",
            "model": CLAUDE_MODEL_ID,
            "system_prompt": "You are a helpful assistant specializing in ETL observability and log analysis.",
            "temperature": 0.0,
            "max_tokens": 1000,
        },
        "actions": [
            {
                "action_type": "predict",
                "method": "POST",
                "headers": {"content-type": "application/json"},
                "url": "https://bedrock-runtime.${parameters.region}.amazonaws.com/model/${parameters.model}/converse",
                "request_body": "{ \"system\": [{\"text\": \"${parameters.system_prompt}\"}], \"messages\": ${parameters.messages}, \"inferenceConfig\": {\"temperature\": ${parameters.temperature}, \"maxTokens\": ${parameters.max_tokens}} }",
            }
        ],
    }

    response = aos_client.transport.perform_request(
        "POST",
        "/_plugins/_ml/connectors/_create",
        body=connector_payload,
    )
    llm_connector_id = response["connector_id"]
else:
    print(f"Connector already exists: {llm_connector_id}")

print(f"LLM connector ID: {llm_connector_id}")

LLM connector ID: 4MWP8J4Bd5wtlVQlFZb0


In [19]:
# Register and deploy the Claude model
if not llm_model_id:
    payload = {
        "name": "Amazon Bedrock Claude (Observability)",
        "function_name": "remote",
        "description": "Claude model for ETL log analysis via OpenSearch ML",
        "connector_id": llm_connector_id,
    }
    response = aos_client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_register",
        body=json.dumps(payload),
        headers={"Content-Type": "application/json"},
    )
    llm_model_id = response["model_id"]
    print(f"Model registered: {llm_model_id}")

    # Deploy
    deploy_response = aos_client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{llm_model_id}/_deploy",
        headers={"Content-Type": "application/json"},
    )
    print(f"Deployment status: {deploy_response['status']}")
else:
    print(f"Model already exists: {llm_model_id}")

Model registered: 5sWP8J4Bd5wtlVQlJJbC
Deployment status: COMPLETED


---
## 6. AI Agent with OpenSearch MCP Server

Build an AI agent that queries OpenSearch logs using natural language via the Model Context Protocol (MCP).

**Architecture:**
```
User Query → Strands Agent → MCP Client → OpenSearch MCP Server → Amazon OpenSearch
                ↓                                                         ↓
        Claude LLM ←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←←← Search Results
```

**Components:**
- **Strands Agents** — Python framework for production-ready AI agents
- **MCP** — Open protocol standardizing agent ↔ tool communication
- **Amazon Bedrock AgentCore** — Serverless runtime for secure, scalable agent deployment

In [20]:
%pip install mcp==1.13.1 strands-agents==1.7.1 uv==0.8.16 -q

# Reload stored variables (in case kernel was restarted)
%store -r OPENSEARCH_URL AWS_ACCESS_KEY_ID AWS_SECRET_ACCESS_KEY AWS_SESSION_TOKEN AWS_REGION MCP_TOKEN_ENDPOINT MCP_ENDPOINT COGNITO_SECRET

Note: you may need to restart the kernel to use updated packages.


### 6.1 Load authentication credentials

Retrieve Cognito client credentials for authenticating with the AgentCore-hosted MCP server.

In [21]:
secrets_client = boto3.client("secretsmanager")
cognito_credentials = json.loads(
    secrets_client.get_secret_value(SecretId=COGNITO_SECRET)["SecretString"]
)

### 6.2 Option A — Local MCP Server (for development)

Launches the OpenSearch MCP server as a local subprocess via `uvx`. Good for fast iteration and debugging.

In [22]:
from mcp import stdio_client, StdioServerParameters
from strands.tools.mcp import MCPClient

mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="uvx",
        args=["opensearch-mcp-server-py"],
        env={
            "OPENSEARCH_URL": OPENSEARCH_URL,
            "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY_ID,
            "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
            "AWS_SESSION_TOKEN": AWS_SESSION_TOKEN,
            "AWS_REGION": AWS_REGION,
        },
    )
))

### 6.3 Option B — AgentCore MCP Server (for production)

Connects to the MCP server deployed on Amazon Bedrock AgentCore via authenticated HTTP transport. Provides session isolation, auto-scaling, and enterprise security.

In [23]:
import requests as http_requests
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

client_id = cognito_credentials["clientId"]
client_secret = cognito_credentials["clientSecret"]
token_endpoint = MCP_TOKEN_ENDPOINT
mcp_url = MCP_ENDPOINT


def get_bearer_token(token_endpoint: str, client_id: str, client_secret: str) -> str:
    """Obtain an OAuth 2.0 access token via client credentials grant."""
    response = http_requests.post(
        token_endpoint,
        data={"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret},
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    response.raise_for_status()
    return response.json()["access_token"]


mcp_client = MCPClient(lambda: streamablehttp_client(
    mcp_url,
    {
        "authorization": f"Bearer {get_bearer_token(token_endpoint, client_id, client_secret)}",
        "Content-Type": "application/json",
    },
    timeout=120,
    terminate_on_close=False,
))

### 6.4 Create and run the AI agents

We create two specialized agents:
- **DevOps Agent** — answers cluster health and shard questions
- **Search Agent** — discovers and analyzes ETL logs for errors and optimization opportunities

In [24]:
from strands import Agent

MODEL = "us.anthropic.claude-sonnet-4-20250514-v1:0"

with mcp_client:
    tools = mcp_client.list_tools_sync()

    devops_agent = Agent(
        tools=[t for t in tools if t.tool_name in ["ClusterHealthTool", "GetShardsTool"]],
        model=MODEL,
        system_prompt=(
            "You are a DevOps agent. Answer questions about OpenSearch cluster health and shards. "
            "When you don't know the answer, ask for more information."
        ),
    )

    search_agent = Agent(
        tools=[t for t in tools if t.tool_name not in ["ClusterHealthTool", "GetShardsTool"]],
        model=MODEL,
        system_prompt="""You are an ETL operations agent. Your role is to:
1. Discover and collect logs related to Glue, MWAA, DAGs, and EC2
2. Identify errors or opportunities to improve those workloads
3. Provide guidance on how to resolve the errors

IMPORTANT RULES:
- To prevent context window overflow, exclude any vector or binary fields from searches.
- Use full-text queries unless specifically asked otherwise.
- When retrieving data from multiple indices, match documents using the ID field if available.
- Use keyword search for exact document ID lookups.
""",
    )

### 6.5 Query your ETL logs

Ask the agent to analyze your logs in natural language.

In [25]:
with mcp_client:
    search_agent("What errors do you see in my logs?")

I'll help you identify errors in your logs related to Glue, MWAA, DAGs, and EC2. Let me start by discovering what log indices are available in your OpenSearch cluster.
Tool #1: ListIndexTool
Great! I can see there's a CloudWatch Logs index (`cwl-2026.06.22`) that likely contains your log data. Let me examine its structure first to understand what kind of logs we're working with.
Tool #2: IndexMappingTool
Perfect! Now let me search for error-related logs in your CloudWatch Logs index. I'll look for common error patterns related to Glue, MWAA, DAGs, and EC2.
Tool #3: SearchIndexTool
Now let me search for more specific details about the Glue job failures to better understand the root cause:
Tool #4: SearchIndexTool
Now let me also check the MWAA task logs to see the specific failures:
Tool #5: SearchIndexTool
Based on my analysis of your logs, I've identified several critical errors in your ETL pipeline. Here's a comprehensive breakdown of the issues and recommendations:

## 🚨 **Critical 